# 05 · Análisis Exploratorio de Datos (EDA) de punta a punta

El EDA no es "correr `.describe()` y graficar histogramas" — es un proceso
sistemático para entender la calidad de los datos, sus patrones y sus riesgos
**antes** de modelar. Este módulo replica el flujo real que seguirías con un
dataset nuevo en el trabajo.

**Dataset:** `credito_solicitudes.csv` (con nulos y duplicados intencionales
para practicar limpieza real).

## Contenido
1. Primer contacto: shape, tipos, memoria
2. Duplicados: detectar y decidir qué hacer
3. Valores nulos: patrón (¿MCAR/MAR/MNAR?) y estrategia de imputación
4. Outliers: IQR, z-score, y por qué el contexto de negocio manda
5. Análisis univariado y bivariado sistemático
6. Feature engineering exploratorio
7. Checklist de reporte de calidad de datos

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

df = pd.read_csv("data/credito_solicitudes.csv", parse_dates=["fecha_solicitud"])
df.shape

(6015, 14)

## 1. Primer contacto

Antes de cualquier análisis: ¿qué tipos tiene cada columna?, ¿cuánta memoria
usa?, ¿los tipos son los que esperarías (fechas como datetime, no como
string)?

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6015 entries, 0 to 6014
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   solicitud_id              6015 non-null   str           
 1   fecha_solicitud           6015 non-null   datetime64[us]
 2   edad                      6015 non-null   int64         
 3   sector                    6015 non-null   str           
 4   antiguedad_laboral_anios  6015 non-null   int64         
 5   antiguedad_empresa_anios  6015 non-null   float64       
 6   ingreso_mensual           5775 non-null   float64       
 7   deuda_actual              6015 non-null   float64       
 8   num_creditos_previos      6015 non-null   int64         
 9   buro_score                5895 non-null   float64       
 10  monto_solicitado          6015 non-null   float64       
 11  plazo_meses               6015 non-null   int64         
 12  tasa_interes              6015 

In [3]:
resumen_columnas = pd.DataFrame({
    "dtype": df.dtypes,
    "n_nulos": df.isna().sum(),
    "pct_nulos": (df.isna().mean() * 100).round(1),
    "n_unicos": df.nunique(),
})
resumen_columnas

,dtype,n_nulos,pct_nulos,n_unicos
solicitud_id,str,0,0.0,6000
fecha_solicitud,datetime64[us],0,0.0,899
edad,int64,0,0.0,58
sector,str,0,0.0,6
antiguedad_laboral_anios,int64,0,0.0,56
antiguedad_empresa_anios,float64,0,0.0,289
ingreso_mensual,float64,240,4.0,5736
deuda_actual,float64,0,0.0,4557
num_creditos_previos,int64,0,0.0,8
buro_score,float64,120,2.0,520


## 2. Duplicados

Un duplicado exacto en un ID que debería ser único es una señal de un bug en
el pipeline de ingesta (doble carga, join mal hecho). Se detecta y se decide
explícitamente si eliminar o investigar el origen.

In [4]:
n_dup_completos = df.duplicated().sum()
n_dup_por_id = df.duplicated(subset="solicitud_id").sum()

print(f"Filas 100% duplicadas: {n_dup_completos}")
print(f"solicitud_id duplicados: {n_dup_por_id}")

if n_dup_por_id > 0:
    print("\nEjemplo de duplicado:")
    id_dup = df.loc[df.duplicated(subset="solicitud_id", keep=False), "solicitud_id"].iloc[0]
    print(df[df["solicitud_id"] == id_dup])

df = df.drop_duplicates(subset="solicitud_id", keep="first")
print(f"\nShape tras deduplicar: {df.shape}")

Filas 100% duplicadas: 15
solicitud_id duplicados: 15

Ejemplo de duplicado:
     solicitud_id fecha_solicitud  edad sector  antiguedad_laboral_anios  antiguedad_empresa_anios  ingreso_mensual  \
573     CR-100205      2023-07-15    37   Agro                        13                       5.7          1314.79   
1265    CR-100205      2023-07-15    37   Agro                        13                       5.7          1314.79   

      deuda_actual  num_creditos_previos  buro_score  monto_solicitado  plazo_meses  tasa_interes  default_90d  
573          393.4                     2       562.0           4467.96           12        0.2273            0  
1265         393.4                     2       562.0           4467.96           12        0.2273            0  

Shape tras deduplicar: (6000, 14)


## 3. Valores nulos: patrón antes que imputación

Antes de imputar hay que entender **por qué** falta el dato:

- **MCAR** (Missing Completely At Random): la ausencia no depende de nada —
  seguro de imputar con media/mediana.
- **MAR** (Missing At Random): la ausencia depende de OTRA variable observada
  — imputar condicionando en esa variable (ej. mediana por grupo).
- **MNAR** (Missing Not At Random): la ausencia depende del propio valor
  faltante (ej. gente con ingresos muy altos no los reporta) — la imputación
  simple sesga el análisis; considera un indicador de "faltante" como feature.

In [5]:
# ¿El nulo en ingreso_mensual se relaciona con otras variables? (evidencia de MAR vs MCAR)
df["ingreso_es_nulo"] = df["ingreso_mensual"].isna()

comparacion = df.groupby("ingreso_es_nulo", observed=True).agg(
    buro_score_prom=("buro_score", "mean"),
    tasa_default=("default_90d", "mean"),
    n=("solicitud_id", "count"),
).round(3)
comparacion

,buro_score_prom,tasa_default,n
ingreso_es_nulo,,,
False,527.449,0.179,5760
True,534.631,0.150,240


In [6]:
# Si no hay diferencia sistemática notable -> tratamos como MCAR/MAR leve: imputar mediana POR SECTOR
# (mejor que mediana global: respeta que sectores tienen niveles de ingreso distintos)
mediana_por_sector = df.groupby("sector", observed=True)["ingreso_mensual"].transform("median")
df["ingreso_mensual_imputado"] = df["ingreso_mensual"].fillna(mediana_por_sector)

# buro_score: mantenemos el nulo explícito -> "sin historial crediticio" es información real,
# no ruido a rellenar (candidato a MNAR: quien no tiene score suele ser cliente nuevo)
df["sin_historial_crediticio"] = df["buro_score"].isna().astype(int)

print(f"Nulos en ingreso_mensual: {df['ingreso_mensual'].isna().sum()} -> {df['ingreso_mensual_imputado'].isna().sum()} tras imputar")
print(f"Solicitudes sin historial crediticio: {df['sin_historial_crediticio'].sum()} ({df['sin_historial_crediticio'].mean():.1%})")

Nulos en ingreso_mensual: 240 -> 0 tras imputar
Solicitudes sin historial crediticio: 120 (2.0%)


## 4. Outliers: IQR, z-score, y criterio de negocio

Un outlier estadístico no siempre es un error de datos — puede ser un cliente
corporativo legítimo. La regla es: **detectar con estadística, decidir con
contexto de negocio.**

In [7]:
def detectar_outliers_iqr(serie, k=1.5):
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    limite_inf, limite_sup = q1 - k * iqr, q3 + k * iqr
    return (serie < limite_inf) | (serie > limite_sup), (limite_inf, limite_sup)

outliers_monto, (lim_inf, lim_sup) = detectar_outliers_iqr(df["monto_solicitado"])
print(f"Límites IQR (k=1.5): (${lim_inf:,.0f}, ${lim_sup:,.0f})")
print(f"Outliers detectados: {outliers_monto.sum()} ({outliers_monto.mean():.1%} de las filas)")

# Z-score modificado (robusto, basado en mediana) -> mejor que z-score clásico con datos sesgados
mediana = df["monto_solicitado"].median()
mad = (df["monto_solicitado"] - mediana).abs().median()  # median absolute deviation
z_robusto = 0.6745 * (df["monto_solicitado"] - mediana) / mad
outliers_z = z_robusto.abs() > 3.5

print(f"Outliers por z-score robusto: {outliers_z.sum()} ({outliers_z.mean():.1%})")
print(f"Coinciden ambos métodos en: {(outliers_monto & outliers_z).sum()} filas")

Límites IQR (k=1.5): ($-13,889, $49,316)
Outliers detectados: 312 (5.2% de las filas)
Outliers por z-score robusto: 231 (3.9%)
Coinciden ambos métodos en: 231 filas


In [8]:
# Contexto de negocio: ¿los outliers de monto son un segmento coherente (empresas grandes)
# o basura de captura (montos absurdos, ej. de más de $50M)?
top_outliers = df.loc[outliers_monto].sort_values("monto_solicitado", ascending=False)
print(top_outliers[["sector", "ingreso_mensual", "monto_solicitado", "buro_score", "antiguedad_empresa_anios"]].head(5))
print("\n-> Los montos altos vienen acompañados de ingresos altos y empresas más establecidas:")
print("   parecen un segmento real (empresas grandes), no errores de captura. NO se eliminan;")
print("   se documentan como segmento aparte para el modelado.")

           sector  ingreso_mensual  monto_solicitado  buro_score  antiguedad_empresa_anios
3537  Manufactura         35709.88         238111.34       850.0                       6.9
4828   Tecnología         17575.22         127483.56       850.0                      13.4
2920   Tecnología         19436.91         125198.49       850.0                       2.5
3592         Agro         15658.36         121405.67       666.0                       6.1
2854  Manufactura         15969.91         120906.22       771.0                      24.1

-> Los montos altos vienen acompañados de ingresos altos y empresas más establecidas:
   parecen un segmento real (empresas grandes), no errores de captura. NO se eliminan;
   se documentan como segmento aparte para el modelado.


## 5. Análisis univariado y bivariado sistemático

Un checklist rápido y repetible en vez de graficar al azar:
- Univariado numérico: distribución (forma, sesgo) por variable.
- Univariado categórico: frecuencias, ¿hay categorías raras con pocos casos?
- Bivariado: cada predictor candidato vs. la variable objetivo.

In [9]:
# Univariado categórico: frecuencias + detección de categorías con muy pocos casos
for col in ["sector", "plazo_meses"]:
    frecuencias = df[col].value_counts(normalize=True).mul(100).round(1)
    print(f"--- {col} ---")
    print(frecuencias)
    categorias_raras = frecuencias[frecuencias < 3]
    if not categorias_raras.empty:
        print(f"  Categorías <3% del total (candidatas a agrupar en 'Otros'): {list(categorias_raras.index)}")
    print()

--- sector ---
sector
Comercio        27.6
Servicios       24.2
Manufactura     17.1
Tecnología      12.3
Construcción    10.0
Agro             8.8
Name: proportion, dtype: float64

--- plazo_meses ---
plazo_meses
12    24.8
24    21.0
18    19.9
6     15.4
36    11.2
48     7.7
Name: proportion, dtype: float64



In [10]:
# Bivariado sistemático: tasa de default por bin de cada variable numérica candidata
variables_numericas = ["buro_score", "ingreso_mensual_imputado", "antiguedad_empresa_anios", "num_creditos_previos"]

for col in variables_numericas:
    bins = pd.qcut(df[col], q=4, duplicates="drop")
    tasa_por_bin = df.groupby(bins, observed=True)["default_90d"].agg(["mean", "count"])
    print(f"--- Tasa de default por cuartil de {col} ---")
    print(tasa_por_bin.round(3))
    print()

--- Tasa de default por cuartil de buro_score ---
                   mean  count
buro_score                    
(299.999, 420.0]  0.560   1483
(420.0, 539.0]    0.129   1471
(539.0, 632.0]    0.015   1458
(632.0, 850.0]    0.004   1468

--- Tasa de default por cuartil de ingreso_mensual_imputado ---
                                 mean  count
ingreso_mensual_imputado                    
(429.10900000000004, 2540.488]  0.211   1500
(2540.488, 3620.64]             0.188   1500
(3620.64, 5119.638]             0.172   1500
(5119.638, 35709.88]            0.141   1500

--- Tasa de default por cuartil de antiguedad_empresa_anios ---
                           mean  count
antiguedad_empresa_anios              
(0.099, 3.8]              0.196   1503
(3.8, 6.6]                0.184   1505
(6.6, 10.3]               0.160   1503
(10.3, 35.6]              0.172   1489

--- Tasa de default por cuartil de num_creditos_previos ---
                       mean  count
num_creditos_previos              

## 6. Feature engineering exploratorio

El EDA no solo detecta problemas — genera hipótesis de features que
alimentarán el módulo de Machine Learning.

In [11]:
df["dti"] = df["deuda_actual"] / (df["ingreso_mensual_imputado"] + 1)
df["monto_sobre_ingreso"] = df["monto_solicitado"] / (df["ingreso_mensual_imputado"] + 1)
df["cuota_mensual_aprox"] = df["monto_solicitado"] * (df["tasa_interes"] / 12) / (1 - (1 + df["tasa_interes"] / 12) ** (-df["plazo_meses"]))
df["carga_deuda_total"] = (df["cuota_mensual_aprox"] + df["deuda_actual"] * 0.03) / (df["ingreso_mensual_imputado"] + 1)

nuevas_features = ["dti", "monto_sobre_ingreso", "cuota_mensual_aprox", "carga_deuda_total"]
correlacion_con_target = df[nuevas_features + ["default_90d"]].corr(numeric_only=True)["default_90d"].drop("default_90d")
print("Correlación de nuevas features con default_90d:")
print(correlacion_con_target.sort_values(ascending=False).round(3))

Correlación de nuevas features con default_90d:
dti                    0.547
carga_deuda_total      0.120
monto_sobre_ingreso    0.062
cuota_mensual_aprox   -0.001
Name: default_90d, dtype: float64


## 7. Checklist de reporte de calidad de datos

Antes de pasar el dataset a modelado, un analista senior documenta:

In [12]:
reporte_calidad = {
    "filas_originales": 6015,
    "filas_tras_deduplicar": len(df),
    "columnas_con_nulos": list(resumen_columnas.loc[resumen_columnas["n_nulos"] > 0].index),
    "estrategia_nulos": {
        "ingreso_mensual": "imputado con mediana por sector",
        "buro_score": "mantenido como nulo + flag 'sin_historial_crediticio'",
    },
    "outliers_detectados_monto": int(outliers_monto.sum()),
    "decision_outliers": "conservados (segmento real de empresas grandes)",
    "features_nuevas": nuevas_features,
    "tasa_default_global": round(df["default_90d"].mean(), 4),
}
import json
print(json.dumps(reporte_calidad, indent=2, ensure_ascii=False))

{
  "filas_originales": 6015,
  "filas_tras_deduplicar": 6000,
  "columnas_con_nulos": [
    "ingreso_mensual",
    "buro_score"
  ],
  "estrategia_nulos": {
    "ingreso_mensual": "imputado con mediana por sector",
    "buro_score": "mantenido como nulo + flag 'sin_historial_crediticio'"
  },
  "outliers_detectados_monto": 312,
  "decision_outliers": "conservados (segmento real de empresas grandes)",
  "features_nuevas": [
    "dti",
    "monto_sobre_ingreso",
    "cuota_mensual_aprox",
    "carga_deuda_total"
  ],
  "tasa_default_global": 0.1782
}


## Resumen y siguientes pasos

- El orden importa: primer contacto → duplicados → nulos (con hipótesis de
  patrón) → outliers (con criterio de negocio) → univariado/bivariado →
  feature engineering.
- Nunca imputes sin antes preguntarte si el nulo mismo es información
  (`sin_historial_crediticio` es más útil que rellenarlo a ciegas).
- Documenta cada decisión de limpieza — el reporte de calidad es el contrato
  entre EDA y modelado.

**Siguiente módulo:** `06_visualizacion_datos.ipynb` — convertir estos
hallazgos en gráficas que comuniquen, no solo que "se vean bonitas".